In [1]:
# Cell 0 — Project Imports

import torch
from torch.nn import functional as F

In [2]:
# Cell 1 — Voxel-Wise Cross-Entropy

def compute_voxelwise_cross_entropy(
    logits: torch.Tensor,  # [B, K, D, H, W], torch.float32
    target: torch.Tensor,  # [B, D, H, W], torch.long
) -> torch.Tensor:         # [B, D, H, W], torch.float32
    """각 voxel의 정답 class에 대한 Cross-Entropy 계산."""

    # 각 voxel의 정답 class에 대한 loss 계산
    # reduction="none"으로 spatial dimension 보존
    voxelwise_cross_entropy = F.cross_entropy(
        input=logits,
        target=target,
        reduction="none",
    )  # [B, D, H, W]

    return voxelwise_cross_entropy


# 3개 class, 4개 voxel의 가상 logits 생성
synthetic_logits = torch.tensor(
    [
        [
            [[[4.0, 0.0], [0.0, 0.0]]],  # Class 0 logits
            [[[0.0, 4.0], [0.0, 2.0]]],  # Class 1 logits
            [[[0.0, 0.0], [4.0, 1.0]]],  # Class 2 logits
        ]
    ],
    dtype=torch.float32,
)  # [B=1, K=3, D=1, H=2, W=2]


# 각 voxel의 정답 class ID 생성
synthetic_target = torch.tensor(
    [[[[0, 1], [2, 1]]]],
    dtype=torch.long,
)  # [B=1, D=1, H=2, W=2]


# Class 축 K에 Softmax를 적용해 probability 계산
class_probabilities = synthetic_logits.softmax(
    dim=1,
)  # [B=1, K=3, D=1, H=2, W=2]


# Target class ID에 해당하는 probability만 선택
target_probabilities = class_probabilities.gather(
    dim=1,
    index=synthetic_target.unsqueeze(dim=1),
).squeeze(
    dim=1,
)  # [B=1, D=1, H=2, W=2]


# PyTorch의 voxel별 Cross-Entropy 계산
voxelwise_cross_entropy = compute_voxelwise_cross_entropy(
    logits=synthetic_logits,
    target=synthetic_target,
)  # [B=1, D=1, H=2, W=2]


# Cross-Entropy 정의를 이용한 직접 계산
manual_cross_entropy = -torch.log(
    target_probabilities,
)  # [B=1, D=1, H=2, W=2]


# 모든 voxel의 loss를 평균해 scalar 생성
mean_cross_entropy = voxelwise_cross_entropy.mean()  # []


print("Logits shape:        ", synthetic_logits.shape)
print("Target shape:        ", synthetic_target.shape)
print("Target probabilities:", target_probabilities)
print("Voxel-wise CE:       ", voxelwise_cross_entropy)
print("Manual CE:           ", manual_cross_entropy)
print("Mean CE:             ", mean_cross_entropy.item())

print(
    "Manual match:        ",
    torch.allclose(
        voxelwise_cross_entropy,
        manual_cross_entropy,
    ),
)

Logits shape:         torch.Size([1, 3, 1, 2, 2])
Target shape:         torch.Size([1, 1, 2, 2])
Target probabilities: tensor([[[[0.9647, 0.9647],
          [0.9647, 0.6652]]]])
Voxel-wise CE:        tensor([[[[0.0360, 0.0360],
          [0.0360, 0.4076]]]])
Manual CE:            tensor([[[[0.0360, 0.0360],
          [0.0360, 0.4076]]]])
Mean CE:              0.1288837492465973
Manual match:         True
